# Plots for the PMLR poster

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

In [ ]:
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import to_rgba

from rewarduq.utils_ext.ml import load_wandb_history, load_wandb_summary
from rewarduq.utils_ext.plot import Plotter
from rewarduq.utils_ext.tools import setup_logging

plt.ioff()
setup_logging()

logger = logging.getLogger(__name__)

PATH_DATA = Path("data")
PATH_OUTPUT = Path("../../output/plots/pmlr_poster")

# setup plotter

FONTSIZE_SMALL = 10
FONTSIZE_DEFAULT = 14
FONTSIZE_LARGE = 16

Plotter.setup(css_patches=["overflow_auto", "gray_background"])
Plotter.configure(
    basewidth=13.7,
    fontsize=FONTSIZE_DEFAULT,
    latex=False,
    rcparams={
        "lines.linewidth": 1,  # default: 1.5
        "axes.labelpad": 6,  # default: 4
    },
    save_dir=PATH_OUTPUT,
    save_format="pdf",
)
Plotter.configure(
    latex=True,
    latex_preamble="\n".join(
        [
            r"\usepackage[utf8]{inputenc}",
            r"\usepackage[T1]{fontenc}",
            r"\usepackage{microtype}",
            r"\usepackage{lmodern}",  # for 8-bit Latin Modern font
            r"\usepackage[sc]{mathpazo}",  # for Palatino font
            r"\usepackage{amsmath,amssymb,amsfonts,mathrsfs}",
        ]
    ),
)

In [ ]:
EXPERIMENT_MAPPING = {
    # "linear_rm": "morning-violet-3",
    "linear_rm": "stellar-cherry-61",
    "enn_rm": "polished-sweep-14",
    # "enn_rm": "confused-sweep-23",
    "dpo_rm": "20250525-165417-intelligent-colt-835",
    "dpo_lora_rm": "20250525-165710-unequaled-lark-148",
}

# DESCRIPTION_MAPPING = {
#     "linear_rm": "Bayesian linear RM",
#     "enn_rm": "Ensemble-based RM",
#     "dpo_rm": "Dropout-based DPO RM",
#     "dpo_lora_rm": "Dropout-based DPO RM (with LoRA)",
# }


def load_calibration_curves(path):
    df_curves = pd.read_csv(path)
    curves = {
        name: tuple(group[key] for key in ["bin", "count", "p_true", "p_pred"])
        for name, group in df_curves.groupby("name")
    }
    return curves

## Main plots

In [ ]:
Plotter.configure(save_always=True)

In [ ]:
def plot_preferences(plot_group):
    # load data
    history_prefs = load_wandb_history(
        PATH_DATA / "prefs.csv",
        {
            "train/epoch": "epoch",
            "eval/prefs/pred_mean": "pred",
            "eval/prefs/lower_mean": "lower",
            "eval/prefs/upper_mean": "upper",
        },
    )

    # plot preferences
    fig, ax = Plotter.create(ncols=4, sharex=True, sharey=True)
    for i, (name, experiment_name) in enumerate(EXPERIMENT_MAPPING.items()):
        ax[i].plot(
            history_prefs[experiment_name]["epoch"],
            history_prefs[experiment_name]["pred"],
            label=name,
        )
        ax[i].fill_between(
            history_prefs[experiment_name]["epoch"],
            history_prefs[experiment_name]["lower"],
            history_prefs[experiment_name]["upper"],
            alpha=0.25,
        )
        Plotter.set(
            ax[i],
            title=name,
        )
    fig.supxlabel("epochs", x=0.525)
    ax[0].set_ylabel("preference probability")
    plot_group.add_plot(fig, "prefs")


with Plotter.group(
    figwidth=1.0,
    grid_ncols=1,
    consistent_size=True,
    save_kw=dict(transparent=True),
    # save=True,
) as plot_group:
    plot_preferences(plot_group)

In [ ]:
def plot_calibration_curves(plot_group):
    def compute_calibration_error(curve, error_fn):
        bins, bin_count, bin_prob_true, bin_prob_pred = curve
        n_samples = np.sum(bin_count)

        calibration_error = error_fn(bin_prob_true, bin_prob_pred)
        ece = np.nansum(bin_count / n_samples * calibration_error).item()
        mce = np.nanmax(calibration_error).item()
        return ece, mce

    def plot_calibration_curve(ax, curve, color="tab:blue", text=None):
        bins, bin_count, bin_prob_true, bin_prob_pred = curve
        n_bins = len(bins) - 1

        bin_count_rel = bin_count / np.max(bin_count)  # normalize by max
        bin_count_rel = np.log(1 + bin_count_rel) / np.log(2)  # apply log scale
        bin_count_rel = 0.1 + 0.9 * np.nan_to_num(bin_count_rel)  # rescale to range [0.1, 1.0]
        bin_colors = [to_rgba(color, alpha) for alpha in bin_count_rel]
        ax.plot([0, 1], [0, 1], linestyle="--", color="tab:gray", label="perfect calibration")
        ax.bar(bins, bin_prob_true, 1 / n_bins, align="edge", color=bin_colors, label="actual calibration")

        if text is not None:
            ax.text(0.05, 0.95, text, fontsize=FONTSIZE_SMALL, va="top", transform=ax.transAxes)

    # load data
    curves_pred = load_calibration_curves(PATH_DATA / "calibration_curve_pred.csv")
    curves_lower = load_calibration_curves(PATH_DATA / "calibration_curve_lower.csv")

    # create plots
    for name, experiment_name in EXPERIMENT_MAPPING.items():
        fig, axes = Plotter.create(ncols=2, sharey=True)

        ece, mce = compute_calibration_error(
            curves_pred[experiment_name], lambda p_true, p_pred: np.abs(p_true - p_pred)
        )
        elce, mlce = compute_calibration_error(
            curves_lower[experiment_name], lambda p_true, p_lower: np.maximum(p_lower - p_true, 0)
        )

        plot_calibration_curve(axes[0], curves_pred[experiment_name], text=f"ECE: {ece:.2f}")
        plot_calibration_curve(axes[1], curves_lower[experiment_name], text=f"ELCE: {elce:.2f}")

        axes[0].set_xlabel("predicted win rate")
        axes[1].set_xlabel("predicted lower bound")
        axes[0].set_ylabel("win rate")

        plot_group.add_plot(fig, f"calibration_curve-{name}")


def plot_calibration_error(plot_group):
    # load data
    df = load_wandb_summary(
        PATH_DATA / "summary.csv",
        {
            "eval/prefs/ece": "ece",
            "eval/prefs/elce": "elce",
        },
    )

    # get rows for the experiments of interest
    df = pd.DataFrame({name: df.loc[experiment_name] for name, experiment_name in EXPERIMENT_MAPPING.items()}).T

    xticks = dict(ticks=range(len(df)), labels=df.index, rotation=25, ha="right")

    # plot ece
    fig, ax = Plotter.create()
    df["ece"].plot.bar(ax=ax, color=to_rgba("tab:red", alpha=0.75))
    Plotter.set(
        ax,
        xlabel="",
        ylabel="ECE",
        xticks=xticks,
    )
    plot_group.add_plot(fig, "ece")

    # plot ece
    fig, ax = Plotter.create()
    df["elce"].plot.bar(ax=ax, color=to_rgba("tab:red", alpha=0.75))
    Plotter.set(
        ax,
        xlabel="",
        ylabel="ELCE",
        xticks=xticks,
    )
    plot_group.add_plot(fig, "elce")


with Plotter.group(
    figwidth=0.33,
    axratio=[1, 1, 0.48, 1, 1, 0.48],
    grid_ncols=3,
    consistent_size=True,
    save_kw=dict(transparent=True),
    # save=True,
) as plot_group:
    plot_calibration_curves(plot_group)
    plot_calibration_error(plot_group)
    plot_group.rearrange([0, 1, 4, 2, 3, 5])

In [ ]:
def plot_accuracy(plot_group):
    # load data
    df = load_wandb_summary(
        PATH_DATA / "summary.csv",
        {
            "eval/win_rate": "win rate",
            "eval/prefs/confident_correct": "confident correct",
            "eval/prefs/ambiguous": "ambiguous",
            "eval/prefs/confident_incorrect": "confident incorrect",
        },
    )

    # get rows for the experiments of interest
    df = pd.DataFrame({name: df.loc[experiment_name] for name, experiment_name in EXPERIMENT_MAPPING.items()}).T

    xticks = dict(ticks=range(len(df)), labels=df.index, rotation=25, ha="right")

    # plot win rates
    fig, ax = Plotter.create()
    df["win rate"].plot.bar(ax=ax, color=to_rgba("tab:green", alpha=0.75))
    Plotter.set(
        ax,
        xlabel="",
        ylabel="win rate",
        ylim=(0.5, 0.8),
        xticks=xticks,
    )
    ax.tick_params(axis="both", labelsize=FONTSIZE_LARGE)
    ax.xaxis.label.set_size(FONTSIZE_LARGE)
    ax.yaxis.label.set_size(FONTSIZE_LARGE)
    plot_group.add_plot(fig, "win_rate")

    # plot confidence statistics
    fig, ax = Plotter.create()
    metrics = ["confident correct", "ambiguous", "confident incorrect"]
    color = [to_rgba(c, alpha=0.75) for c in ["tab:green", "tab:blue", "tab:red"]]
    df[metrics].plot.bar(ax=ax, stacked=True, color=color)
    Plotter.set(
        ax,
        xlabel="",
        xticks=xticks,
        legend=dict(loc="lower right", fontsize=FONTSIZE_LARGE),
    )
    ax.tick_params(axis="both", labelsize=FONTSIZE_LARGE)
    ax.xaxis.label.set_size(FONTSIZE_LARGE)
    ax.yaxis.label.set_size(FONTSIZE_LARGE)
    plot_group.add_plot(fig, "confidence_statistics")


with Plotter.group(
    figwidth=0.5,
    grid_ncols=2,
    consistent_size=True,
    save_kw=dict(transparent=True),
    # save=True,
) as plot_group:
    plot_accuracy(plot_group)

In [ ]:
Plotter.configure(save_always=False)
plt.close()

## temp

In [ ]:
def plot_preference_distributions(plot_group):
    # load data
    history_prefs = load_wandb_history(
        PATH_DATA / "prefs_distribution.csv",
        {
            "train/epoch": "epoch",
            "eval/prefs/lower_min": "min",
            "eval/prefs/lower_p10": "p10",
            "eval/prefs/pred_median": "median",
            "eval/prefs/upper_p90": "p90",
            "eval/prefs/upper_max": "max",
        },
    )

    # plot preferences
    fig, ax = Plotter.create(ncols=4, sharex=True, sharey=True)
    for i, (name, experiment_name) in enumerate(EXPERIMENT_MAPPING.items()):
        ax[i].plot(
            history_prefs[experiment_name]["epoch"],
            history_prefs[experiment_name]["median"],
            label=name,
        )
        ax[i].fill_between(
            history_prefs[experiment_name]["epoch"],
            history_prefs[experiment_name]["p10"],
            history_prefs[experiment_name]["p90"],
            alpha=0.25,
        )
        Plotter.set(
            ax[i],
            title=name,
        )
    fig.supxlabel("epochs", x=0.525)
    ax[0].set_ylabel("preference probability")
    plot_group.add_plot(fig, "preferece_distribution")


with Plotter.group(
    figwidth=1.0,
    grid_ncols=1,
    consistent_size=True,
    save_kw=dict(transparent=True),
    # save=True,
) as plot_group:
    plot_preference_distributions(plot_group)

In [ ]:
def plot_uncertainties(plot_group):
    # load data
    history_uncertainty = load_wandb_history(
        PATH_DATA / "prefs_uncertainty.csv",
        {
            "train/epoch": "epoch",
            "eval/prefs/uncertainty_mean": "uncertainty",
        },
    )

    # plot uncertainties
    fig, ax = Plotter.create()
    for name, experiment_name in EXPERIMENT_MAPPING.items():
        ax.plot(
            history_uncertainty[experiment_name]["epoch"],
            history_uncertainty[experiment_name]["uncertainty"],
            label=name,
        )
        # history[experiment_name].plot.line(x="epoch", y="uncertainty", ax=ax, label=name)
    Plotter.set(
        ax,
        ylabel="uncertainty",
        legend=True,
    )
    plot_group.add_plot(fig, "uncertainty")


with Plotter.group(
    figwidth=0.5,
    grid_ncols=1,
    consistent_size=True,
    save_kw=dict(transparent=True),
    # save=True,
) as plot_group:
    plot_uncertainties(plot_group)

In [ ]:
def plot_uncertainty_distributions(plot_group):
    # load data
    history_prefs = load_wandb_history(
        PATH_DATA / "prefs_uncertainty_distribution.csv",
        {
            "train/epoch": "epoch",
            "eval/prefs/uncertainty_min": "min",
            "eval/prefs/uncertainty_p10": "p10",
            "eval/prefs/uncertainty_median": "median",
            "eval/prefs/uncertainty_p90": "p90",
            "eval/prefs/uncertainty_max": "max",
        },
    )

    # plot preferences
    fig, ax = Plotter.create(ncols=4, sharex=True, sharey=True)
    for i, (name, experiment_name) in enumerate(EXPERIMENT_MAPPING.items()):
        ax[i].plot(
            history_prefs[experiment_name]["epoch"],
            history_prefs[experiment_name]["median"],
            label=name,
        )
        ax[i].fill_between(
            history_prefs[experiment_name]["epoch"],
            history_prefs[experiment_name]["p10"],
            history_prefs[experiment_name]["p90"],
            alpha=0.25,
        )
        Plotter.set(
            ax[i],
            title=name,
        )
    fig.supxlabel("epochs", x=0.525)
    ax[0].set_ylabel("uncertainty")
    plot_group.add_plot(fig, "uncertainty_distribution")


with Plotter.group(
    figwidth=1.0,
    grid_ncols=1,
    consistent_size=True,
    save_kw=dict(transparent=True),
    # save=True,
) as plot_group:
    plot_uncertainty_distributions(plot_group)